<h2>Food (v0.03) - 'Food (v0.03).ipynb'</h2>

<u><h5>Context</h5></u>

The notebook aims to isolate NZF impacts modelled for individual commodities with 'CPC' codes forwarded over by David Forengie on 23rd September 2026.

The objectives of the notebook are:

<ol>
    <li>
        To develop the representation of CPC codes provided by David, as its not completely clear how these work (e.g. where some CPC codes are only 3 digits, whereas others are 5). 
    </li>
    <li>
        To read-in 'v0.64' of the 'Impact' analysis results for the country in question, then isolate these records for the CPC codes identified in the step above.
    </li>
</ol>

<u><h5>Version Documentation</h5></u>

'v0.01' outlined an initial method for completing the process. 'v0.02' refined the method and developed a fully functioning notebook, utilising results from the 'Impact' notebooks which now include CPC codes as fields in the output data tables. Upon inspection, it was found that the identification of CPC codes wasn't functioning properly, where CPC codes specified by David wouldn't show in the dataset, yet HSa4 codes given in the CPC manual would show. This likely explains much of the losses observed when pulling by CPC codes, so 'v0.03' utilises a new version of the codes sent through David which have been mapped to the HS system ahead of time. 

In [1]:
# Import packages
import os
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def download_link(df, title="Download CSV file", filename="data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

# define function for counting values in each data field
trade_im_field_count = lambda field: trade_im[field].value_counts().to_frame().reset_index()
trade_ex_field_count = lambda field: trade_ex[field].value_counts().to_frame().reset_index()

version = "v0.03"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/food/"

# 1. Set up the notebook

## 1.1 Let's set the country we're interested in and set up a results folder.

In [2]:
# Select one of our countries to work on...
iso_code = "180"
iso_country = "Congo, Democratic Republic of the"
year = 2019
import_portfolio = "h4_ag6_by_sea_M.csv" # Imports, '_M' | Foreign Imports, '_FM' | Re-Imports, '_RM'
export_portfolio = "h4_ag6_by_sea_X.csv" # Exports, '_X' | Domestic Exports, '_DX' | Re-exports, '_RX'
gdp = 46.850*1e9

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."

## 1.2 Import and Format 'Impact Analysis' Results

In [3]:
impact_im_c = [
    "trade_idx", "reporterCode", "reporterDesc", "partnerCode", "partnerDesc", "flowDesc", "motDesc", 
    "HS6", "HS6_desc", "HS4", "HS4_desc", "HS2", "HS2_desc", "cpc", "cpc_desc", "eora_desc", "gtap", "gtap_desc", 
    "cifvalue", "fobvalue", "primaryValue", "qtyUnitAbbr", "qty", "netWgt", "vol_kg_vtw", 
    "sea_dist_km", "sea_dist_nm", "tnm", "fac_vessel_types", "fac_vessel_types_str", 
    #"TandT_match_desc", "TandT_ad_val_pct", "TandT_N", "TandT_ciffob", "TandT_cifvalue", "TandT_fobvalue", "cost_int_match_desc",
    "bau_usd_26", "levy_usd_26", "asis_usd_26", "brazil_usd_26", "liscr_usd_26", "direct_usd_26", 
    "bau_usd_29", "levy_usd_29", "asis_usd_29", "brazil_usd_29", "liscr_usd_29", "direct_usd_29", 
    "bau_usd_32", "levy_usd_32", "asis_usd_32", "brazil_usd_32", "liscr_usd_32", "direct_usd_32", 
    "bau_usd_35", "levy_usd_35", "asis_usd_35", "brazil_usd_35", "liscr_usd_35", "direct_usd_35", 
    "bau_usd_38", "levy_usd_38", "asis_usd_38", "brazil_usd_38", "liscr_usd_38", "direct_usd_38", 
    "bau_usd_41", "levy_usd_41", "asis_usd_41", "brazil_usd_41", "liscr_usd_41", "direct_usd_41", 
    "bau_usd_44", "levy_usd_44", "asis_usd_44", "brazil_usd_44", "liscr_usd_44", "direct_usd_44", 
    "bau_usd_47", "levy_usd_47", "asis_usd_47", "brazil_usd_47", "liscr_usd_47", "direct_usd_47", 
    "bau_usd_50", "levy_usd_50", "asis_usd_50", "brazil_usd_50", "liscr_usd_50", "direct_usd_50"	
]

impact_im_d = {
    "reporterCode":"str", "partnerCode":"str", 
    "HS6":"str", "HS4":"str", "HS2":"str", "HS2":"str", "gtap":"str", "cpc":"str"
}

impact_im = pd.read_csv(
    input_dir + "impact/{0}/impact_im_v0.64.csv".format(iso_country), 
    dtype=impact_im_d,
    usecols=impact_im_c
)[impact_im_c]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(impact_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(impact_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(impact_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(impact_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(impact_im.vol_kg_vtw.sum() / 1e6, 1)))

impact_im["cpc_cl"] = [str(i).split('.')[0] for i in impact_im.cpc.values]
impact_im = impact_im.drop(columns="trade_idx").reset_index(drop=True).reset_index().rename(columns={"index":"trade_idx"})
display(impact_im.head(2))

Aggregate trade portfolio statistics for 'Congo, Democratic Republic of the': 

Number of trade records present in dataset: 13906
Number of partner countries present in dataset: 121
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$1475.4m
Total weight within the dataset: 1445.3 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl
0,0,180,Dem. Rep. of the Congo,699,India,Import,Sea,630130,630130 | Blankets (other than,6301,6301 | Blankets and travell,63,"63 | Textiles, made up ar",2711,Blankets and travelling rugs (except electric ...,Textiles and Wearing Apparel,27,TEX - Textiles,638.337,0,638.337,kg,1950.0,1950.0,116.049667,12865.0,7993.937915,927.693830,['Container'],['Container'],4.848897,4.840431,4.841258,4.841258,4.845052,4.845052,5.072193,8.989517,5.594791,5.194243,5.208350,5.338797,4.966568,8.803856,5.888871,5.394118,5.270333,5.509027,4.873929,9.573596,7.302482,6.386494,5.908715,5.742038,4.783007,10.179079,8.376213,7.849222,6.326040,6.154028,4.693801,10.315071,9.855137,9.939873,6.858085,6.464608,4.602807,10.103055,9.681822,10.607859,7.846560,6.583094,4.517084,9.012171,9.393081,9.842968,7.378600,6.660876,4.429674,8.278449,8.415260,10.496951,8.671954,6.704985,2711
1,1,180,Dem. Rep. of the Congo,276,Germany,Import,Sea,630190,630190 | Blankets and travell,6301,6301 | Blankets and travell,63,"63 | Textiles, made up ar",2711,Blankets and travelling rugs (except electric ...,Textiles and Wearing Apparel,27,TEX - Textiles,1742.769,0,1742.769,kg,2065.0,2065.0,316.835404,9292.0,5773.779332,1829.337708,['Container'],['Container'],9.561636,9.544941,9.546572,9.546572,9.554054,9.554054,10.001957,17.726606,11.032479,10.242630,10.270447,10.527679,9.793673,17.360496,11.612380,10.636767,10.392674,10.863359,9.610997,18.878363,14.399909,12.593652,11.651512,11.322837,9.431706,20.072326,16.517219,15.478036,12.474443,12.135248,9.255799,20.340491,19.433539,19.600630,13.523592,12.747687,9.076366,19.922414,19.091776,20.917846,15.472786,12.981332,8.907327,17.771276,18.522400,19.409543,14.550007,13.134713,8.734961,16.324436,16.594216,20.699143,17.100397,13.221692,2711


# 2. Subset the 'Impact Analysis' results for food security-related CPC codes

## 2.1 Let's get a sense of how much of import value is associated with food-relevant goods across classification systems

### 2.1.1 Harmonised System (HS01-23)

In [4]:
#print("HS groupings: \n\n{0}\n".format(pd.Series(impact_im.HS2_desc.unique()).sort_values().values))
HSa2_good_groups = [
    "01", "02", "03", "04", "05", "07", "08", # "06", 
    "09", "10", "11", "12", "13", "14", "15", "16", 
    "17", "18", "19", "20", "21", "22", "23"
]

impact_im_HSa2_food = impact_im[(impact_im.HS2.isin(HSa2_good_groups))]
impact_im_HSa2_food_g = impact_im_HSa2_food.groupby(["HS2_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_HSa2_food_g["pct_val"] = round(100 * impact_im_HSa2_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_HSa2_food_g["pct_vol"] = round(100 * impact_im_HSa2_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_HSa2_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_HSa2_food_g.round(1)
))

Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): $498.3m (33.8%), ... 

                              cifvalue   vol_kg_vtw  pct_val  pct_vol
HS2_desc                                                             
10 | Cereals               185620078.2  603427448.1     12.6     41.8
02 | Meat and edible meat   89229841.3   46326929.1      6.0      3.2
03 | Fish and crustaceans   45673735.2   16399243.3      3.1      1.1
21 | Miscellaneous edible   23389543.3    7650095.5      1.6      0.5
15 | Animal, vegetable or   21766534.6   24206582.9      1.5      1.7
11 | Products of the mill   21753223.7   41680930.0      1.5      2.9
19 | Preparations of cere   20289492.7    7844864.1      1.4      0.5
17 | Sugars and sugar con   19828892.2   28831344.0      1.3      2.0
22 | Beverages, spirits a   14569382.7   25504957.3      1.0      1.8
20 | Preparations of vege   11952857.1   10075071.1      0.8      0.7
16 | Meat, fish, crustace   10413685.2    2287935.1      0.

### 2.1.2 EORA groupings ('Food & Beverages' and 'Fishing')

In [5]:
print("EORA groupings: \n\n{0}\n".format(pd.Series(impact_im.eora_desc.unique()).sort_values().values))
eora_good_groups = ["Food & Beverages", "Agriculture", "Fishing"]

impact_im_eora_food = impact_im[(impact_im.eora_desc.isin(eora_good_groups))]
impact_im_eora_food_g = impact_im_eora_food.groupby(["eora_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_eora_food_g["pct_val"] = round(100 * impact_im_eora_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_eora_food_g["pct_vol"] = round(100 * impact_im_eora_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_eora_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_eora_food_g.round(1)
))

EORA groupings: 

['Agriculture' 'Electrical and Machinery' 'Fishing' 'Food & Beverages'
 'Metal Products' 'Mining and Quarrying' 'Other Manufacturing'
 'Petroleum, Chemical and Non-Metallic Mineral Products'
 'Textiles and Wearing Apparel' 'Transport Equipment' 'Wood and Paper' nan]

Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): $335.7m (22.8%), ... 

                     cifvalue   vol_kg_vtw  pct_val  pct_vol
eora_desc                                                   
Agriculture       214053740.4  259410628.6     14.5     17.9
Food & Beverages  105788024.1   77336383.7      7.2      5.4
Fishing            15834879.2    2019128.2      1.1      0.1



### 2.1.3 GTAP (16x headings)

In [6]:
#print("GTAP groupings: \n\n{0}\n".format(pd.Series(impact_im.gtap_desc.unique()).sort_values().values))
gtap_food_groups = [
    "GRO - Cereal grains n.e.c.", "PDR - Paddy rice", "PCR - Processed rice", "SGR - Sugar", 
    "CMT - Bovine meat prods", "CTL - Bovine cattle, sheep and goats, horses", "FSH - Fishing", "OMT - Meat products n.e.c.",
    "MIL - Dairy products", "OFD - Food products n.e.c.", "V_F - Vegetables, fruit, nuts", "OSD - Oil seeds", 
    "OCR - Crops n.e.c.", "PFB - Plant-based fibers", "VOL - Vegetable oils and fats", 
    "OAP - Animal products n.e.c."
]

impact_im_gtap_food = impact_im[(impact_im.gtap_desc.isin(gtap_food_groups))]
impact_im_gtap_food_g = impact_im_gtap_food.groupby(["gtap_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_gtap_food_g["pct_val"] = round(100 * impact_im_gtap_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_gtap_food_g["pct_vol"] = round(100 * impact_im_gtap_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via GTAP: ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_gtap_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_gtap_food_g.head(15).round(1)
))

Value of food imports identified via GTAP: $303.7m (20.6%), ... 

                                  cifvalue  vol_kg_vtw  pct_val  pct_vol
gtap_desc                                                               
OFD - Food products n.e.c.     101701489.4  53872558.7      6.9      3.7
OMT - Meat products n.e.c.      74925586.3  38222457.8      5.1      2.6
PCR - Processed rice            37638359.1  65377829.7      2.6      4.5
VOL - Vegetable oils and fats   26353287.1  33916223.8      1.8      2.3
CMT - Bovine meat prods         18620790.4   9098901.0      1.3      0.6
GRO - Cereal grains n.e.c.      12806939.4  52974563.5      0.9      3.7
V_F - Vegetables, fruit, nuts   10178297.5  15269877.8      0.7      1.1
MIL - Dairy products            10101199.0   4666115.8      0.7      0.3
SGR - Sugar                      5422521.8  11927023.8      0.4      0.8
PDR - Paddy rice                 2903157.1   5042783.9      0.2      0.3
OAP - Animal products n.e.c.     2000030.9    472560.4    

## 2.2 Comparison of results

In [7]:
print("Share of total import value associated with food-relevant goods by classification system...\n\nHSa2 {0}% \nGTAP {1}% \nEORA {2}%".format(
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))

Share of total import value associated with food-relevant goods by classification system...

HSa2 33.8% 
GTAP 20.6% 
EORA 22.8%


# 3. Take HSa4 codes associated with specific CPCs

## 3.1 Set HSa4 Codes we're interested in

In [8]:
hsa4_codes_food = {
    "Dairy": [
        "0401", "0402", "0403", "0404", "0405", "0406",
    ],
    "Fats and Oils": [
        "1501", "1502", "1503", "1504", "1505", "1506", "1507", "1508", "1509", "1510"
        "1511", "1512", "1513", "1514", "1515", "1516", "1517"
    ],
    "Fruits": [
        "0709", "0801", "0802", "0803", "0804", "0805", "0806", "0807", "0808", "0809", 
        "0810", "0813", "2009"
    ],
    "Meat and Prep": [
        "0201", "0203", "0204", "0205", "0207", "0208", "0209", "2301"
    ],
    "Processed Cereals": [
        "1006", "1101", "1102", "1104", "1105", "1106", "1107", "1108", "1109", "1302",
        "1602", "1901", "1902", "1903", "1904", "1905", "2005", "2007", "2104", "2106", 
        "3505"
    ],
    "Sugar and Honey": [
        "1701", "1702", "1703", "1704", "2006"
    ],
    "Unprocessed Cereals": [
        "1001", "1002", "1003", "1004", "1005", "1007", "1008"
    ],
    "Vegetables": [
        "0701", "0702", "0703", "0704", "0705", "0706", "0707", "0708", "0709", "0710",
        "0712", "0713", "0714", "1209", "2009"
    ]
}

## 3.2 Identify 'Food Security' groups of interest via CPC

In [9]:
impact_im["food_group"] = np.nan

for food_group in list(hsa4_codes_food.keys()):
    impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group

display(impact_im.head(2))

/var/folders/_k/81z8ryjd28lc4f0bb4dm7gkw0000gn/T/ipykernel_46366/1412186970.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Dairy' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl,food_group
0,0,180,Dem. Rep. of the Congo,699,India,Import,Sea,630130,630130 | Blankets (other than,6301,6301 | Blankets and travell,63,"63 | Textiles, made up ar",2711,Blankets and travelling rugs (except electric ...,Textiles and Wearing Apparel,27,TEX - Textiles,638.337,0,638.337,kg,1950.0,1950.0,116.049667,12865.0,7993.937915,927.693830,['Container'],['Container'],4.848897,4.840431,4.841258,4.841258,4.845052,4.845052,5.072193,8.989517,5.594791,5.194243,5.208350,5.338797,4.966568,8.803856,5.888871,5.394118,5.270333,5.509027,4.873929,9.573596,7.302482,6.386494,5.908715,5.742038,4.783007,10.179079,8.376213,7.849222,6.326040,6.154028,4.693801,10.315071,9.855137,9.939873,6.858085,6.464608,4.602807,10.103055,9.681822,10.607859,7.846560,6.583094,4.517084,9.012171,9.393081,9.842968,7.378600,6.660876,4.429674,8.278449,8.415260,10.496951,8.671954,6.704985,2711,NaN
1,1,180,Dem. Rep. of the Congo,276,Germany,Import,Sea,630190,630190 | Blankets and travell,6301,6301 | Blankets and travell,63,"63 | Textiles, made up ar",2711,Blankets and travelling rugs (except electric ...,Textiles and Wearing Apparel,27,TEX - Textiles,1742.769,0,1742.769,kg,2065.0,2065.0,316.835404,9292.0,5773.779332,1829.337708,['Container'],['Container'],9.561636,9.544941,9.546572,9.546572,9.554054,9.554054,10.001957,17.726606,11.032479,10.242630,10.270447,10.527679,9.793673,17.360496,11.612380,10.636767,10.392674,10.863359,9.610997,18.878363,14.399909,12.593652,11.651512,11.322837,9.431706,20.072326,16.517219,15.478036,12.474443,12.135248,9.255799,20.340491,19.433539,19.600630,13.523592,12.747687,9.076366,19.922414,19.091776,20.917846,15.472786,12.981332,8.907327,17.771276,18.522400,19.409543,14.550007,13.134713,8.734961,16.324436,16.594216,20.699143,17.100397,13.221692,2711,NaN


## 3.3 Inspect the output

In [10]:
impact_im_cpc_food = impact_im[(~impact_im.food_group.isna())]
impact_im_cpc_food_g = impact_im_cpc_food.groupby("food_group").agg({"HS6_desc":"count", "cifvalue":"sum", "vol_kg_vtw":"sum"}).reset_index().sort_values(by="cifvalue", ascending=False).round(2).rename(columns={"HS6_desc":"N_com"}).reset_index(drop=True)
impact_im_cpc_food_g["pct_val"] = round(100 * impact_im_cpc_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_cpc_food_g["pct_vol"] = round(100 * impact_im_cpc_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Proportion of imports identified as food security-relevant:\n")
print("Value: ${0}m of ${1}m ({2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / 1e6, 1),
    round(impact_im.cifvalue.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))
print("Volume: {0} of {1} million kilograms ({2}%)".format(
    round(impact_im_cpc_food.vol_kg_vtw.sum() / 1e6, 1),
    round(impact_im.vol_kg_vtw.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.vol_kg_vtw.sum() / impact_im.vol_kg_vtw.sum(), 1)
))
print("Value-to-Weight: {0} from {1} US$ per kg (+{2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum(), 2),
    round(impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum(), 2),
    round(100 * ((impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum()) - (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum())) / (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum()), 1)
))
print("Average distance to market (volume-weighted): {0} from {1} nm (+{2}%)".format(
    round((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000), 1),
    round((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000), 1),
    round(100 * (((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000)) - ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000))) / \
          ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000)), 1)
))
print("\n{0}\n".format(impact_im_cpc_food_g.head(15).round(1)))

impact_im_cpc_food_g.to_csv(output_dir + "{0}/food_total_usd_{1}.csv".format(iso_country, version), index=False)

Proportion of imports identified as food security-relevant:

Value: $360.4m of $1475.4m (24.4%)
Volume: 746.2 of 1445.3 million kilograms (51.6%)
Value-to-Weight: 0.48 from 1.02 US$ per kg (+-52.7%)
Average distance to market (volume-weighted): 6688.0 from 6355.8 nm (+5.2%)

            food_group  N_com     cifvalue   vol_kg_vtw  pct_val  pct_vol
0  Unprocessed Cereals     26  145078562.0  533006834.5      9.8     36.9
1    Processed Cereals    347  100340901.4  124428462.3      6.8      8.6
2        Meat and Prep     75   70617935.0   37224774.2      4.8      2.6
3      Sugar and Honey     60   19828892.2   28831344.0      1.3      2.0
4           Vegetables    110   10887606.1   15497345.8      0.7      1.1
5                Dairy    114    9707951.6    4273349.4      0.7      0.3
6        Fats and Oils     71    2520797.1    1552655.8      0.2      0.1
7               Fruits     51    1389539.0    1395515.8      0.1      0.1



## 3.3 Derive results for 2029, 2041 and 2050

### 3.3.1 Total FEM costs by scenario (US$)

In [11]:
print("\nTotal FEM costs by scenario (US$)...\n")

d = {"food_group": list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["bau", "levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            round(100 * impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum(), 1) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$)...



,food_group,bau_29,levy_29,asis_29,brazil_29,liscr_29,direct_29,bau_41,levy_41,asis_41,brazil_41,liscr_41,direct_41,bau_50,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,1.501965e+07,2.638047e+07,1.651174e+07,1.535223e+07,1.539308e+07,1.579070e+07,1.391845e+07,3.042229e+07,2.900876e+07,2.928218e+07,2.021198e+07,1.906106e+07,1.314978e+07,2.437197e+07,2.483467e+07,3.067551e+07,2.543778e+07,1.975942e+07
1,Fats and Oils,5.468694e+06,9.417348e+06,5.961769e+06,5.557016e+06,5.572113e+06,5.734675e+06,5.080476e+06,1.108416e+07,1.047941e+07,1.058964e+07,7.282883e+06,6.879791e+06,4.809490e+06,8.736578e+06,9.008130e+06,1.077716e+07,9.083743e+06,7.123799e+06
2,Fruits,2.851684e+06,4.997870e+06,3.130641e+06,2.923613e+06,2.930080e+06,2.997634e+06,2.643242e+06,5.742678e+06,5.455908e+06,5.500271e+06,3.815062e+06,3.619613e+06,2.497744e+06,4.682308e+06,4.747729e+06,5.829204e+06,4.758763e+06,3.754976e+06
3,Meat and Prep,1.187995e+08,2.105500e+08,1.310397e+08,1.216581e+08,1.219885e+08,1.250438e+08,1.099369e+08,2.415968e+08,2.308243e+08,2.328090e+08,1.606282e+08,1.514123e+08,1.037506e+08,1.938956e+08,1.970999e+08,2.458567e+08,2.031121e+08,1.570423e+08
4,Processed Cereals,5.160554e+08,8.958362e+08,5.650266e+08,5.264325e+08,5.278369e+08,5.416137e+08,4.793399e+08,1.034993e+09,9.879992e+08,9.999906e+08,6.934421e+08,6.502522e+08,4.537115e+08,8.334043e+08,8.463713e+08,1.043114e+09,8.648145e+08,6.732853e+08
5,Sugar and Honey,7.583340e+07,1.167187e+08,7.970950e+07,7.576144e+07,7.593865e+07,7.834897e+07,7.185385e+07,1.403706e+08,1.333427e+08,1.377523e+08,9.835220e+07,8.963952e+07,6.907603e+07,1.154181e+08,1.165697e+08,1.335080e+08,1.130823e+08,9.190501e+07
6,Unprocessed Cereals,1.816631e+09,3.195650e+09,1.998356e+09,1.857615e+09,1.862650e+09,1.910091e+09,1.683391e+09,3.676795e+09,3.511649e+09,3.546452e+09,2.451278e+09,2.305676e+09,1.590386e+09,2.953633e+09,3.001426e+09,3.727802e+09,3.084004e+09,2.389894e+09
7,Vegetables,6.890648e+07,1.210145e+08,7.578581e+07,7.048612e+07,7.066837e+07,7.244399e+07,6.386951e+07,1.389589e+08,1.327558e+08,1.340549e+08,9.272585e+07,8.741947e+07,6.035338e+07,1.119434e+08,1.137427e+08,1.411164e+08,1.166176e+08,9.062050e+07


### 3.3.2 Total FEM costs by scenario (% cif)

In [12]:
print("\nTotal FEM costs by scenario (% cif)...\n")

d = {"food_group":list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            np.round(100 * (impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum() - impact_im[(impact_im.food_group == hsa4_codes_food_key)]["bau_usd_{1}".format(scenario, year)].sum()) / \
                     impact_im[(impact_im.food_group == hsa4_codes_food_key)].cifvalue.sum(), 2) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (% cif)...



,food_group,levy_29,asis_29,brazil_29,liscr_29,direct_29,levy_41,asis_41,brazil_41,liscr_41,direct_41,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,1.17,0.15,0.03,0.04,0.08,1.70,1.55,1.58,0.65,0.53,1.16,1.20,1.81,1.27,0.68
1,Fats and Oils,1.57,0.20,0.04,0.04,0.11,2.38,2.14,2.19,0.87,0.71,1.56,1.67,2.37,1.70,0.92
2,Fruits,1.54,0.20,0.05,0.06,0.11,2.23,2.02,2.06,0.84,0.70,1.57,1.62,2.40,1.63,0.90
3,Meat and Prep,1.30,0.17,0.04,0.05,0.09,1.86,1.71,1.74,0.72,0.59,1.28,1.32,2.01,1.41,0.75
4,Processed Cereals,3.78,0.49,0.10,0.12,0.25,5.54,5.07,5.19,2.13,1.70,3.78,3.91,5.87,4.10,2.19
5,Sugar and Honey,2.06,0.20,-0.00,0.01,0.13,3.46,3.10,3.32,1.34,0.90,2.34,2.40,3.25,2.22,1.15
6,Unprocessed Cereals,9.51,1.25,0.28,0.32,0.64,13.74,12.60,12.84,5.29,4.29,9.40,9.73,14.73,10.30,5.51
7,Vegetables,4.79,0.63,0.15,0.16,0.32,6.90,6.33,6.45,2.65,2.16,4.74,4.90,7.42,5.17,2.78
